# Part B – Question 1: Random Forest from Scratch

We implement a **Decision Tree** (for regression) and a **Random Forest** from scratch using only NumPy, then evaluate both on the sklearn diabetes dataset.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes  # data loading only
import matplotlib.pyplot as plt

## 1. Load and Split Data

In [ ]:
# Load diabetes regression dataset (442 samples, 10 features)
data = load_diabetes()
X, y = data.data, data.target

# 80/20 train-test split (manually, no sklearn)
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Decision Tree Class (Regression, CART-style)

We use **variance reduction** (equivalent to minimising MSE) as the splitting criterion.

In [ ]:
class Node:
    """A single node in the decision tree."""
    def __init__(self):
        self.feature = None       # feature index to split on
        self.threshold = None     # split threshold
        self.left = None          # left child Node
        self.right = None         # right child Node
        self.value = None         # leaf prediction (mean of y)


class DecisionTree:
    """
    Regression Decision Tree built with a greedy (CART) strategy.

    Hyperparameters
    ---------------
    max_depth       : maximum depth of the tree (None = unlimited)
    min_samples     : minimum number of samples required to split a node
    feature_indices : if given, only consider these feature columns (used by RandomForest)
    """

    def __init__(self, max_depth=None, min_samples=2, feature_indices=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.feature_indices = feature_indices
        self.root = None

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def build_tree(self, x: list, y: list):
        """Build the tree on training data x (n x d) and targets y (n,)."""
        X = np.array(x, dtype=float)
        Y = np.array(y, dtype=float)
        self.root = self._build(X, Y, depth=0)

    def predict(self, x: list) -> float:
        """Return predicted value for a single data point x (d,)."""
        x = np.array(x, dtype=float)
        return self._traverse(self.root, x)

    def predict_batch(self, X):
        """Convenience: predict for all rows of X."""
        return np.array([self.predict(row) for row in X])

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build(self, X, y, depth):
        node = Node()

        # Stopping conditions → make leaf
        if (len(y) < self.min_samples or
                (self.max_depth is not None and depth >= self.max_depth) or
                np.var(y) == 0):
            node.value = float(np.mean(y))
            return node

        # Find best split
        feat, thresh = self._best_split(X, y)

        if feat is None:          # no valid split found
            node.value = float(np.mean(y))
            return node

        # Partition
        left_mask = X[:, feat] <= thresh
        right_mask = ~left_mask

        node.feature = feat
        node.threshold = thresh
        node.left  = self._build(X[left_mask],  y[left_mask],  depth + 1)
        node.right = self._build(X[right_mask], y[right_mask], depth + 1)
        return node

    def _best_split(self, X, y):
        """Return (feature_index, threshold) of the split that minimises weighted MSE."""
        best_loss = float('inf')
        best_feat, best_thresh = None, None

        # Which features to consider
        features = self.feature_indices if self.feature_indices is not None else range(X.shape[1])

        for feat in features:
            col = X[:, feat]
            thresholds = np.unique(col)
            for thresh in thresholds:
                left_y  = y[col <= thresh]
                right_y = y[col  > thresh]
                if len(left_y) == 0 or len(right_y) == 0:
                    continue
                loss = (len(left_y)  * np.var(left_y) +
                        len(right_y) * np.var(right_y)) / len(y)
                if loss < best_loss:
                    best_loss  = loss
                    best_feat  = feat
                    best_thresh = thresh

        return best_feat, best_thresh

    def _traverse(self, node, x):
        """Recursively descend the tree for a single point x."""
        if node.value is not None:           # leaf
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(node.left, x)
        return self._traverse(node.right, x)


print("DecisionTree class defined.")

## 3. Random Forest Class

In [ ]:
class RandomForest:
    """
    Random Forest for regression.

    Each tree is trained on:
      - a bootstrap sample of the rows (with replacement)
      - a random subset of sqrt(d) features

    Prediction is the mean of all tree predictions.
    """

    def __init__(self, n_trees=50, max_depth=None, min_samples=2, seed=42):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.seed = seed
        self.trees = []

    def build_forest(self, x: list, y: list):
        """Train n_trees decision trees on random subsets of data and features."""
        X = np.array(x, dtype=float)
        Y = np.array(y, dtype=float)
        n_samples, n_features = X.shape
        n_feat_subset = max(1, int(np.sqrt(n_features)))  # sqrt(d) features per tree

        rng = np.random.default_rng(self.seed)
        self.trees = []

        for _ in range(self.n_trees):
            # Bootstrap: sample rows with replacement
            row_idx = rng.integers(0, n_samples, size=n_samples)
            # Random feature subset
            feat_idx = rng.choice(n_features, size=n_feat_subset, replace=False)

            X_sub = X[row_idx]
            y_sub = Y[row_idx]

            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples=self.min_samples,
                feature_indices=feat_idx
            )
            tree.build_tree(X_sub, y_sub)
            self.trees.append(tree)

    def predict(self, x: list) -> float:
        """Predict for a single point: average of all tree predictions."""
        return float(np.mean([t.predict(x) for t in self.trees]))

    def predict_batch(self, X):
        """Predict for all rows of X."""
        return np.array([self.predict(row) for row in X])


print("RandomForest class defined.")

## 4. Train and Evaluate

In [ ]:
def mse(y_true, y_pred):
    return float(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

# ---- Single Decision Tree (unrestricted depth) ----
print("Training Decision Tree...")
dt = DecisionTree(max_depth=None, min_samples=2)
dt.build_tree(X_train, y_train)

dt_train_pred = dt.predict_batch(X_train)
dt_test_pred  = dt.predict_batch(X_test)
dt_train_mse  = mse(y_train, dt_train_pred)
dt_test_mse   = mse(y_test,  dt_test_pred)
print(f"Decision Tree  | Train MSE: {dt_train_mse:.2f}  | Test MSE: {dt_test_mse:.2f}")

# ---- Random Forest (50 trees) ----
print("Training Random Forest (50 trees)...")
rf = RandomForest(n_trees=50, max_depth=None, min_samples=2, seed=42)
rf.build_forest(X_train, y_train)

rf_train_pred = rf.predict_batch(X_train)
rf_test_pred  = rf.predict_batch(X_test)
rf_train_mse  = mse(y_train, rf_train_pred)
rf_test_mse   = mse(y_test,  rf_test_pred)
print(f"Random Forest  | Train MSE: {rf_train_mse:.2f}  | Test MSE: {rf_test_mse:.2f}")

## 5. Visualise Predictions vs Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(axes,
                             [dt_test_pred, rf_test_pred],
                             ['Decision Tree', 'Random Forest']):
    ax.scatter(y_test, preds, alpha=0.6, s=30)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', label='Perfect prediction')
    ax.set_xlabel('True value')
    ax.set_ylabel('Predicted value')
    ax.set_title(f'{title}\nTest MSE = {mse(y_test, preds):.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('q1_predictions.png', dpi=100)
plt.show()
print("Plot saved.")

## 6. Effect of Number of Trees on MSE

In [ ]:
tree_counts = [1, 5, 10, 20, 50, 100]
test_mses = []

for n in tree_counts:
    rf_n = RandomForest(n_trees=n, max_depth=None, min_samples=2, seed=42)
    rf_n.build_forest(X_train, y_train)
    preds = rf_n.predict_batch(X_test)
    test_mses.append(mse(y_test, preds))

plt.figure(figsize=(7, 4))
plt.plot(tree_counts, test_mses, marker='o')
plt.axhline(dt_test_mse, color='r', linestyle='--', label=f'Single Tree ({dt_test_mse:.0f})')
plt.xlabel('Number of trees')
plt.ylabel('Test MSE')
plt.title('Random Forest: Test MSE vs Number of Trees')
plt.legend()
plt.tight_layout()
plt.savefig('q1_ntrees_mse.png', dpi=100)
plt.show()